In [29]:
import pandas as pd
df=pd.read_csv("Data_sheet_week1.csv")
df.sample(5)
df.isnull()
df.isnull().sum()
df=df.ffill()
df


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,FREESHIP,Facebook,1325.06
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


In [27]:
import numpy as np
from scipy import stats

In [30]:
# 1. Calculate Q1, Q3, and IQR
Q1 = df1['TotalPrice'].quantile(0.25)
Q3 = df1['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

# 2. Define Lower and Upper Bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Identify Outliers (Optional: just to see them)
outliers_iqr = df1[(df1['TotalPrice'] < lower_bound) | (df1['TotalPrice'] > upper_bound)]
print(f"Number of IQR outliers: {len(outliers_iqr)}")

# --- NEUTRALIZATION OPTIONS ---

# Option A: Neutralize by Trimming (Dropping the outlier rows entirely)
df_clean_iqr_trimmed = df1[(df1['TotalPrice'] >= lower_bound) & (df1['TotalPrice'] <= upper_bound)].copy()

# Option B: Neutralize by Capping (Replacing extreme values with the boundary values)
df_clean_iqr_capped = df1.copy()

# Apply upper cap
df_clean_iqr_capped.loc[df_clean_iqr_capped['TotalPrice'] > upper_bound, 'TotalPrice'] = upper_bound
# Apply lower cap
df_clean_iqr_capped.loc[df_clean_iqr_capped['TotalPrice'] < lower_bound, 'TotalPrice'] = lower_bound

print(f"Trimming shape: {df_clean_iqr_trimmed.shape} | Capping shape: {df_clean_iqr_capped.shape}")

Number of IQR outliers: 8
Trimming shape: (1192, 14) | Capping shape: (1200, 14)


In [31]:
# 3 New Predictive Features

# 1. Feature 1: Average Value Per Item (Ratio Feature)

df['AvgValuePerItem'] = df['TotalPrice'] / (df['ItemsInCart'] + 1e-5)

# 2. Feature 2: Is Discounted (Boolean/Binary Feature)

expected_price = df['Quantity'] * df['UnitPrice']
df['Is_Discounted'] = (expected_price > df['TotalPrice']).astype(int)

# 3. Feature 3: Log Transformation of Total Price (Statistical Transformation)

df['Log_TotalPrice'] = np.log1p(df['TotalPrice'])

# 4. BONUS Feature: High Value Cart Flag (Threshold/Binning Feature)

top_25_threshold = df['TotalPrice'].quantile(0.75)
df['Is_High_Value'] = (df['TotalPrice'] > top_25_threshold).astype(int)

# Check our new features
print(df[['TotalPrice', 'ItemsInCart', 'AvgValuePerItem', 'Is_Discounted', 'Log_TotalPrice', 'Is_High_Value']].head())

   TotalPrice  ItemsInCart  AvgValuePerItem  Is_Discounted  Log_TotalPrice  \
0     2853.10            7       407.585132              0        7.956512   
1      302.70            3       100.899664              0        5.716040   
2     2753.40            8       344.174570              0        7.920955   
3      273.19            5        54.637891              0        5.613821   
4     2504.04            8       313.004609              0        7.826060   

   Is_High_Value  
0              1  
1              0  
2              1  
3              0  
4              1  
